# SetWise — Multi-Dataset IMU Rep Counter v4 (Raw Signal)

**Primary task**: Rep counting (L1 regression)
**Auxiliary task**: Exercise classification (weight 0.1 — regularises encoder only)

**Architecture**: Single-branch raw-signal (CNN-Attention-TCN with padding masks)
- Inputs padded/truncated to MAX_LEN=3072 samples — rep count preserved in true temporal structure
- ConvStem (EEGNet-style) → Multi-head self-attention (with key_padding_mask) → Dilated causal TCN×2 → Masked avg pool

**Contrast with v3 (time-normalised)**:
- v3 resamples all inputs to T_NORM=512: rep count encoded as cycle frequency
- v4 pads/truncates to MAX_LEN=3072: rep count preserved in raw signal duration and cycle count
- Circular time-shift augmentation removed (not valid for raw-length signals)

**Two-phase training**:
1. Pretrain on recofit (forearm) — learn general rep-counting patterns from large, diverse dataset
2. Fine-tune on Whales1and2 (wrist) — adapt to wrist placement (deployment domain)

**Key dataset decisions**:
- Whales1and2 `has_reps=True`: rep counts 2–30, wrist sensor — primary deployment target
- recofit `has_reps=True` (annotated): 1 011 sets, range 1–61, median 20 — pre-training backbone
- INSIGHT-LME excluded: per-rep time-normalised before assembly → excluded from rep loss
- mm-fit excluded: 92% of sets exactly 10 reps — too narrow for rep regression

In [ ]:
import numpy as np
import pandas as pd
import os
import gc
import warnings
from collections import Counter

import scipy.io
from scipy.signal import resample as scipy_resample
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, mean_absolute_error

import pathlib

# DATA_ROOT is set by the download cell above.
# If running on Colab, override here:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_ROOT = '/content/drive/MyDrive/0SetWise (1)'
    os.listdir('/content/drive/MyDrive')
    os.listdir(DATA_ROOT)
except Exception:
    pass  # DATA_ROOT already set by download cell

print(f'DATA_ROOT: {DATA_ROOT}')
print(f'  Whales1and2 exists : {os.path.isdir(os.path.join(DATA_ROOT, "Whales1and2"))}')
print(f'  recofit exists     : {os.path.isdir(os.path.join(DATA_ROOT, "recofit"))}')

warnings.filterwarnings('ignore')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)

In [ ]:
## ── Download datasets from Google Drive to JupyterHub ────────────────────────
# Fill in the IDs from your Drive share links before running.
#
# How to get IDs:
#   Folder  → right-click folder in Drive → Share → copy link →
#             the ID is the string after /folders/ in the URL
#   File    → right-click file → Get link → ID is between /d/ and /view
#
# Run this cell once; skip on subsequent runs if data is already present.

import subprocess, sys, os

WHALES_FOLDER_ID  = 'PASTE_WHALES1AND2_FOLDER_ID_HERE'
RECOFIT_FILE_ID   = 'PASTE_RECOFIT_MAT_FILE_ID_HERE'

DATA_ROOT = os.path.expanduser('~/0SetWise')
whales_dir  = os.path.join(DATA_ROOT, 'Whales1and2')
recofit_dir = os.path.join(DATA_ROOT, 'recofit')
recofit_mat = os.path.join(recofit_dir, 'exercise_data.50.0000_singleonly.mat')

os.makedirs(whales_dir,  exist_ok=True)
os.makedirs(recofit_dir, exist_ok=True)

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'gdown'])
import gdown

if not os.listdir(whales_dir):
    print('Downloading Whales1and2 folder...')
    gdown.download_folder(id=WHALES_FOLDER_ID, output=whales_dir, quiet=False)
else:
    print(f'Whales1and2 already present ({len(os.listdir(whales_dir))} files)')

if not os.path.exists(recofit_mat):
    print('Downloading recofit .mat file (~1.5 GB, takes a few minutes)...')
    gdown.download(id=RECOFIT_FILE_ID, output=recofit_mat, quiet=False)
else:
    print(f'recofit already present ({os.path.getsize(recofit_mat) // 1_000_000} MB)')

print(f'\nDATA_ROOT: {DATA_ROOT}')
print(f'  Whales1and2 : {os.path.isdir(whales_dir)}  ({len(os.listdir(whales_dir))} files)')
print(f'  recofit     : {os.path.exists(recofit_mat)}')

## Configuration & Exercise Taxonomy

In [ ]:
BASE     = pathlib.Path(DATA_ROOT)
NPZ_PATH = BASE / 'INSIGHT-LME' / 'INSIGHT_LME_processed.npz'

print(f'BASE     : {BASE}')
print(f'NPZ_PATH : {NPZ_PATH} (exists={NPZ_PATH.exists()})')

TARGET_HZ   = 50
MAX_LEN     = 3072   # pad/truncate to this length; covers ~60s sets at 50 Hz
N_CHANNELS  = 6      # acc XYZ + gyro XYZ
MIN_SAMPLES = 5      # low threshold — recofit pushes all classes well above this

REP_WEIGHT  = 1.0    # rep regression is primary
CLS_WEIGHT  = 0.1    # classification is auxiliary encoder regulariser

# ── Exercise taxonomy ─────────────────────────────────────────────────────────
EXERCISE_MAP = {
    # ── INSIGHT-LME (wrist) ──────────────────────────────────────────────────
    'Bicep Curls':          'bicep_curls',
    'Lateral Raise':        'lateral_raises',
    'Lunges':               'lunges',
    'Squats':               'squats',
    'Triceps Extension':    'tricep_extensions',
    'Frontal Raise':        'frontal_raise',

    # ── mm-fit (wrist) ───────────────────────────────────────────────────────
    'squats':                   'squats',
    'lunges':                   'lunges',
    'bicep_curls':              'bicep_curls',
    'situps':                   'situps',
    'pushups':                  'pushups',
    'tricep_extensions':        'tricep_extensions',
    'dumbbell_rows':            'rows',
    'jumping_jacks':            'jumping_jacks',
    'dumbbell_shoulder_press':  'shoulder_press',
    'lateral_shoulder_raises':  'lateral_raises',

    # ── Whales1and2 (wrist) ──────────────────────────────────────────────────
    'APULL':  'pullups',
    'CGCR':   'rows',
    'CGOCTE': 'tricep_extensions',
    'DLR':    'lateral_raises',
    'DSP':    'shoulder_press',
    'IDBC':   'bicep_curls',
    'AIDBC':  'bicep_curls',
    'MGTBR':  'rows',
    'MIBP':   'bench_press',
    'MSP':    'shoulder_press',
    'MTE':    'tricep_extensions',
    'NGCR':   'rows',
    'PREC':   'bicep_curls',
    'SACLR':  'lateral_raises',
    'SAOCTE': 'tricep_extensions',
    'SAODTE': 'tricep_extensions',
    '30BP':   'bench_press',
    '30DBP':  'bench_press',
    '45DBP':  'bench_press',

    # ── recofit (forearm — pretrain only) ────────────────────────────────────
    'Bicep Curl':                                           'bicep_curls',
    'Biceps Curl (band)':                                   'bicep_curls',
    'Two-arm Dumbbell Curl (both arms, not alternating)':   'bicep_curls',
    'Alternating Dumbbell Curl':                            'bicep_curls',
    'Lateral Raise':                                        'lateral_raises',
    'Shoulder Press (dumbbell)':                            'shoulder_press',
    'Squat Rack Shoulder Press':                            'shoulder_press',
    'Overhead Triceps Extension':                           'tricep_extensions',
    'Overhead Triceps Extension (label spans both arms)':   'tricep_extensions',
    'Triceps Kickback (knee on bench) (label spans both arms)': 'tricep_extensions',
    'Triceps Kickback (knee on bench) (left arm)':          'tricep_extensions',
    'Triceps Kickback (knee on bench) (right arm)':         'tricep_extensions',
    'Triceps extension (lying down)':                       'tricep_extensions',
    'Triceps extension (lying down) (left arm)':            'tricep_extensions',
    'Triceps extension (lying down) (right arm)':           'tricep_extensions',
    'Squat':                                                'squats',
    'Squat (arms in front of body, parallel to ground)':    'squats',
    'Squat (hands behind head)':                            'squats',
    'Squat (kettlebell / goblet)':                          'squats',
    'Dumbbell Squat (hands at side)':                       'squats',
    'Squat Jump':                                           'squats',
    'Wall Squat':                                           'squats',
    'Lunge (alternating both legs, weight optional)':       'lunges',
    'Walking lunge':                                        'lunges',
    'Dumbbell Row (knee on bench) (label spans both arms)': 'rows',
    'Dumbbell Row (knee on bench) (left arm)':              'rows',
    'Dumbbell Row (knee on bench) (right arm)':             'rows',
    'Band Pull-Down Row':                                   'rows',
    'Dumbbell Deadlift Row':                                'rows',
    'Pushup (knee or foot variation)':                      'pushups',
    'Pushups':                                              'pushups',
    'Sit-up (hands positioned behind head)':                'situps',
    'Sit-ups':                                              'situps',
    'Butterfly Sit-up':                                     'situps',
    'Crunch':                                               'situps',
    'Jumping Jacks':                                        'jumping_jacks',
    'Chest Press (rack)':                                   'bench_press',
    'Burpee':                                               'burpee',
}

# ── MyoGym integer → canonical name ──────────────────────────────────────────
MYOGYM_LABEL_MAP = {
    1:  'rows',              2:  'rows',              3:  'rows',
    4:  'rows',              5:  'rows',              6:  'rows',
    7:  'rows',              8:  'bench_press',       9:  None,
    10: 'bench_press',       11: None,                12: 'pushups',
    13: 'bench_press',       14: 'bench_press',       15: 'tricep_extensions',
    16: 'tricep_extensions', 17: 'tricep_extensions', 18: 'tricep_extensions',
    19: 'tricep_extensions', 20: 'bicep_curls',       21: 'bicep_curls',
    22: 'bicep_curls',       23: 'bicep_curls',       24: 'bicep_curls',
    25: 'bicep_curls',       26: 'rows',              27: 'lateral_raises',
    28: 'frontal_raise',     29: 'shoulder_press',    30: None,
}

## Dataset Loaders

In [31]:
def _ds(arr, src_hz):
    return arr[:: src_hz // TARGET_HZ] if src_hz != TARGET_HZ else arr


def load_insight(base=BASE):
    """Load preprocessed INSIGHT-LME segments from INSIGHT_LME_processed.npz.
    Generated by INSIGHT-LME/INSIGHT_LME_Processing.ipynb.
    Segmented per set (participant × exercise × trial): ~2383 segments, reps 5–15.
    Arrays are raw 50 Hz (unscaled); prep_norm handles scaling + T_NORM resampling.
    has_reps=False: data was per-rep time-normalised before assembly; cycle frequency
    in T_NORM window does NOT reliably encode rep count, so excluded from rep loss."""
    path = pathlib.Path(NPZ_PATH)
    if not path.exists():
        raise FileNotFoundError(
            f'INSIGHT_LME_processed.npz not found at {path}. '
            "Place it in BASE/INSIGHT-LME/ or /content/drive/MyDrive/Data/."
        )
    data = np.load(path, allow_pickle=True)
    canonical = set(EXERCISE_MAP.values())
    segs = []
    for i in range(len(data['reps'])):
        ex = str(data['exercises'][i])
        if ex not in canonical:
            continue
        segs.append({
            'imu':      data['X'][i].astype(np.float32),
            'exercise': ex,
            'reps':     float(data['reps'][i]),
            'source':   'insight',
            'has_reps': False,
        })
    print(f'INSIGHT-LME  : {len(segs):>5} segments  (classification only)')
    return segs


def load_mmfit(base=BASE):
    """mm-fit, 50 Hz left-wrist.
    has_reps=False: 92% of sets are exactly 10 reps — too narrow for rep regression."""
    mmdir = os.path.join(base, 'mm-fit')
    if not os.path.isdir(mmdir):
        print('mm-fit       :     0 segments (directory not found)')
        return []
    segs  = []
    for w in range(21):
        pdir = os.path.join(mmdir, f'w{w:02d}')
        lp   = os.path.join(pdir, f'w{w:02d}_labels.csv')
        ap   = os.path.join(pdir, f'w{w:02d}_sw_l_acc.npy')
        gp   = os.path.join(pdir, f'w{w:02d}_sw_l_gyr.npy')
        if not all(os.path.exists(p) for p in [lp, ap, gp]): continue
        labels = pd.read_csv(lp, header=None,
                             names=['start_frame','end_frame','reps','exercise_name'])
        acc, gyr = np.load(ap), np.load(gp)
        for _, row in labels.iterrows():
            ex = str(row['exercise_name'])
            if ex not in EXERCISE_MAP: continue
            s, e = int(row['start_frame']), int(row['end_frame'])
            imu  = np.concatenate([acc[s:e, 2:5], gyr[s:e, 2:5]], axis=1).astype(np.float32)
            if len(imu) < 20: continue
            segs.append({'imu': imu, 'exercise': EXERCISE_MAP[ex],
                         'reps': np.nan, 'source': 'mmfit', 'has_reps': False})
    print(f'mm-fit       : {len(segs):>5} segments')
    return segs


def load_whales(base=BASE):
    """Whales1and2, 100 Hz wrist. Rep counts 2–30; primary real-time rep regression source."""
    wdir     = os.path.join(base, 'Whales1and2')
    if not os.path.isdir(wdir):
        print('Whales1and2  :     0 segments (directory not found)')
        return []
    acc_cols = ['wristMotion_accelerationX','wristMotion_accelerationY','wristMotion_accelerationZ']
    gyr_cols = ['wristMotion_rotationRateX', 'wristMotion_rotationRateY', 'wristMotion_rotationRateZ']
    segs     = []
    for fname in sorted(os.listdir(wdir)):
        if not fname.endswith('.csv'): continue
        df = pd.read_csv(os.path.join(wdir, fname)).dropna(subset=acc_cols + gyr_cols)
        if len(df) < 50: continue
        ex = str(df['activity'].iloc[0])
        if ex not in EXERCISE_MAP: continue
        imu  = _ds(df[acc_cols + gyr_cols].values.astype(np.float32), 100)
        segs.append({'imu': imu, 'exercise': EXERCISE_MAP[ex],
                     'reps': float(df['reps'].iloc[0]),
                     'source': 'whales', 'has_reps': True})
    print(f'Whales1and2  : {len(segs):>5} segments')
    return segs


def load_recofit(base=BASE):
    """recofit, 50 Hz forearm arm-band (PRETRAIN ONLY). 1 011 annotated in-taxonomy sets,
    rep range 1–61. Bug fix: recofit uses -1 as sentinel for non-exercise rows; guard r > 0."""
    recofit_dir = os.path.join(base, 'recofit')
    # Force Drive FUSE to populate the directory listing before checking file existence.
    # Without this, os.path.exists() can return False on large files that haven't been listed yet.
    if os.path.isdir(recofit_dir):
        _ = os.listdir(recofit_dir)
    path = os.path.join(recofit_dir, 'exercise_data.50.0000_singleonly.mat')
    if not os.path.exists(path):
        print('recofit      :     0 segments (file not found)')
        return []
    segs = []
    print('Loading recofit (~3 min)...')
    m          = scipy.io.loadmat(path, struct_as_record=False, squeeze_me=True)
    activities = list(m['exerciseConstants'].activities)
    sd         = m['subject_data']
    for i in range(sd.shape[0]):
        for j in range(sd.shape[1]):
            cell = sd[i, j]
            if isinstance(cell, (float, np.floating)) and cell == 0: continue
            ex_name = activities[j]
            if ex_name not in EXERCISE_MAP: continue
            recs = [cell] if not hasattr(cell, '__len__') else cell
            for rec in recs:
                try:
                    imu  = np.concatenate([
                        rec.data.accelDataMatrix[:, 1:4],
                        rec.data.gyroDataMatrix[:,  1:4]
                    ], axis=1).astype(np.float32)
                    reps = float(getattr(rec, 'activityReps', float('nan')))
                    if len(imu) < 20: continue
                    has_reps = not np.isnan(reps) and reps > 0
                    segs.append({'imu': imu, 'exercise': EXERCISE_MAP[ex_name],
                                 'reps': reps if has_reps else np.nan,
                                 'source': 'recofit', 'has_reps': has_reps})
                except Exception:
                    continue
    del m, sd; gc.collect()
    print(f'recofit      : {len(segs):>5} segments')
    return segs


def load_myogym(base=BASE):
    """MyoGym, ~60 Hz Myo Armband forearm — PRETRAIN ONLY, no rep annotations."""
    path = os.path.join(base, 'MyoGym', 'data', 'MyoGym', 'MyoGym.mat')
    if not os.path.exists(path):
        print('MyoGym       :     0 segments (file not found)')
        return []
    print('Loading MyoGym...')
    m         = scipy.io.loadmat(path, struct_as_record=False, squeeze_me=True)
    raw       = m['raw_data']
    label_col = m['raw_data_labels'][:, 0].astype(int)
    del m; gc.collect()

    boundaries = np.concatenate([[0],
                                  np.where(np.diff(label_col) != 0)[0] + 1,
                                  [len(label_col)]])
    segs = []
    for k in range(len(boundaries) - 1):
        start, end = boundaries[k], boundaries[k + 1]
        lbl = label_col[start]
        if lbl not in MYOGYM_LABEL_MAP: continue
        ex_name = MYOGYM_LABEL_MAP[lbl]
        if ex_name is None: continue
        imu = raw[start:end, [10, 11, 12, 14, 15, 16]].astype(np.float32)
        if len(imu) < 50: continue
        segs.append({'imu': imu, 'exercise': ex_name,
                     'reps': np.nan, 'source': 'myogym', 'has_reps': False})

    print(f'MyoGym       : {len(segs):>5} segments')
    return segs

## Load All Datasets

Wrist sensors (Phase 2 fine-tuning): INSIGHT-LME, mm-fit, Whales1and2
Forearm sensors (Phase 1 pretraining only): recofit, MyoGym

In [32]:
# Wrist fine-tune target: Whales only (wrist sensor, real rep counts, deployment domain)
# INSIGHT-LME excluded: per-rep time-normalised → cycle frequency doesn't encode rep count
# mm-fit excluded: 92% of sets are exactly 10 reps — too narrow for rep regression
wrist_segments    = load_whales()
recofit_segments  = load_recofit()
myogym_segments   = load_myogym()
pretrain_segments = recofit_segments + myogym_segments
all_segments      = wrist_segments + pretrain_segments

print(f'\nWrist (Whales): {len(wrist_segments)}   '
      f'Recofit: {len(recofit_segments)}   '
      f'MyoGym: {len(myogym_segments)}   '
      f'Total: {len(all_segments)}')

Whales1and2  :   152 segments
Loading recofit (~3 min)...
recofit      :  1073 segments
MyoGym       :     0 segments (file not found)

Wrist (Whales): 152   Recofit: 1073   MyoGym: 0   Total: 1225


## Exercise Frequency Analysis & Filtering

Keep only classes with ≥ MIN_SAMPLES across the combined dataset.

In [33]:
ex_counts = Counter(s['exercise'] for s in all_segments)

print('Exercise distribution (combined, before filtering):')
for ex, cnt in sorted(ex_counts.items(), key=lambda x: -x[1]):
    wrist_cnt    = sum(1 for s in wrist_segments    if s['exercise'] == ex)
    pretrain_cnt = sum(1 for s in pretrain_segments if s['exercise'] == ex)
    flag = '  <- DROP' if cnt < MIN_SAMPLES else ''
    print(f'  {ex:<32} {cnt:>5}  (wrist {wrist_cnt:>4} | pretrain {pretrain_cnt:>4}){flag}')

keep_exercises    = {ex for ex, cnt in ex_counts.items() if cnt >= MIN_SAMPLES}
wrist_segments    = [s for s in wrist_segments    if s['exercise'] in keep_exercises]
pretrain_segments = [s for s in pretrain_segments if s['exercise'] in keep_exercises]

print(f'\nKept {len(keep_exercises)} classes: {sorted(keep_exercises)}')
print(f'Wrist: {len(wrist_segments)}  Pretrain: {len(pretrain_segments)}')

Exercise distribution (combined, before filtering):
  squats                             241  (wrist    0 | pretrain  241)
  tricep_extensions                  166  (wrist   30 | pretrain  136)
  rows                               134  (wrist   21 | pretrain  113)
  situps                             132  (wrist    0 | pretrain  132)
  bicep_curls                        125  (wrist   24 | pretrain  101)
  shoulder_press                      88  (wrist   10 | pretrain   78)
  pushups                             69  (wrist    0 | pretrain   69)
  lunges                              68  (wrist    0 | pretrain   68)
  bench_press                         56  (wrist   25 | pretrain   31)
  lateral_raises                      51  (wrist   16 | pretrain   35)
  burpee                              44  (wrist    0 | pretrain   44)
  pullups                             26  (wrist   26 | pretrain    0)
  jumping_jacks                       25  (wrist    0 | pretrain   25)

Kept 13 classes: ['bench

## Preprocessing

- All inputs scaled then padded/truncated to MAX_LEN=3072 — raw temporal structure preserved.
- Padding mask M tracks real vs padded timesteps (True=real, False=padding).
- Scaler fit on raw wrist train timesteps to preserve true signal statistics.
- `has_reps=False` segments excluded from rep loss.

In [ ]:
le          = LabelEncoder().fit(sorted(keep_exercises))
label_names = le.classes_
print('Classes:', label_names)

def encode_reps(segs):
    """Fill NaN reps with median of annotated values; NaN entries never hit the loss."""
    r      = np.array([s['reps'] for s in segs], dtype=np.float32)
    median = float(np.nanmedian(r[~np.isnan(r)])) if np.any(~np.isnan(r)) else 10.0
    return np.where(np.isnan(r), median, r)

# ── Wrist: stratified 70/15/15 split ─────────────────────────────────────────
yw   = le.transform([s['exercise'] for s in wrist_segments]).astype(np.int64)
idxw = np.arange(len(yw))
idxw_tv, idxw_test = train_test_split(idxw, test_size=0.15, random_state=42, stratify=yw)
idxw_tr, idxw_val  = train_test_split(idxw_tv, test_size=0.15/0.85, random_state=42,
                                       stratify=yw[idxw_tv])
print(f'Wrist split — train: {len(idxw_tr)}, val: {len(idxw_val)}, test: {len(idxw_test)}')

# ── Scaler: fit on raw wrist train timesteps ──────────────────────────────────
flat_wrist_tr = np.concatenate([wrist_segments[i]['imu'] for i in idxw_tr])
scaler        = StandardScaler().fit(flat_wrist_tr)

def prep_raw(segs, idxs):
    """Scale then pad/truncate each segment to MAX_LEN. Returns (X, M).
    X: (N, MAX_LEN, 6)  M: (N, MAX_LEN) bool, True=real data, False=padding."""
    X, M = [], []
    for i in idxs:
        imu = scaler.transform(segs[i]['imu']).astype(np.float32)
        L   = len(imu)
        if L >= MAX_LEN:
            imu  = imu[:MAX_LEN]
            mask = np.ones(MAX_LEN, dtype=bool)
        else:
            pad  = np.zeros((MAX_LEN - L, imu.shape[1]), dtype=np.float32)
            imu  = np.concatenate([imu, pad], axis=0)
            mask = np.array([True] * L + [False] * (MAX_LEN - L), dtype=bool)
        X.append(imu); M.append(mask)
    return np.stack(X), np.stack(M)

Xw_tr,  Mw_tr  = prep_raw(wrist_segments, idxw_tr)
Xw_val, Mw_val = prep_raw(wrist_segments, idxw_val)
Xw_te,  Mw_te  = prep_raw(wrist_segments, idxw_test)
rw_tr  = encode_reps([wrist_segments[i] for i in idxw_tr])
rw_val = encode_reps([wrist_segments[i] for i in idxw_val])
rw_te  = encode_reps([wrist_segments[i] for i in idxw_test])
yw_tr, yw_val, yw_te = yw[idxw_tr], yw[idxw_val], yw[idxw_test]

hrw_tr  = np.array([wrist_segments[i]['has_reps'] for i in idxw_tr],  dtype=bool)
hrw_val = np.array([wrist_segments[i]['has_reps'] for i in idxw_val], dtype=bool)
hrw_te  = np.array([wrist_segments[i]['has_reps'] for i in idxw_test], dtype=bool)
print(f'Wrist train — rep-annotated: {hrw_tr.sum()}/{len(hrw_tr)}  '
      f'val: {hrw_val.sum()}/{len(hrw_val)}  '
      f'test: {hrw_te.sum()}/{len(hrw_te)}')

# ── Pretrain (recofit + MyoGym): optional ───────────────────────────────────
has_pretrain = len(pretrain_segments) > 0
if has_pretrain:
    yp   = le.transform([s['exercise'] for s in pretrain_segments]).astype(np.int64)
    idxp = np.arange(len(yp))
    idxp_tr, idxp_val = train_test_split(idxp, test_size=0.15, random_state=42, stratify=yp)
    print(f'Pretrain split — train: {len(idxp_tr)}, val: {len(idxp_val)}')

    Xp_tr,  Mp_tr  = prep_raw(pretrain_segments, idxp_tr)
    Xp_val, Mp_val = prep_raw(pretrain_segments, idxp_val)
    rp_tr  = encode_reps([pretrain_segments[i] for i in idxp_tr])
    rp_val = encode_reps([pretrain_segments[i] for i in idxp_val])
    yp_tr, yp_val = yp[idxp_tr], yp[idxp_val]

    hrp_tr  = np.array([pretrain_segments[i]['has_reps'] for i in idxp_tr],  dtype=bool)
    hrp_val = np.array([pretrain_segments[i]['has_reps'] for i in idxp_val], dtype=bool)
    print(f'Pretrain train — rep-annotated: {hrp_tr.sum()}/{len(hrp_tr)}  '
          f'val: {hrp_val.sum()}/{len(hrp_val)}')
else:
    Xp_tr  = np.empty((0, MAX_LEN, N_CHANNELS), dtype=np.float32)
    Xp_val = np.empty((0, MAX_LEN, N_CHANNELS), dtype=np.float32)
    Mp_tr  = np.empty((0, MAX_LEN), dtype=bool)
    Mp_val = np.empty((0, MAX_LEN), dtype=bool)
    rp_tr  = np.empty((0,), dtype=np.float32)
    rp_val = np.empty((0,), dtype=np.float32)
    yp_tr  = np.empty((0,), dtype=np.int64)
    yp_val = np.empty((0,), dtype=np.int64)
    hrp_tr = np.empty((0,), dtype=bool)
    hrp_val= np.empty((0,), dtype=bool)
    print('Pretrain split — skipped (no recofit/MyoGym loaded)')

print(f'\nXw_tr: {Xw_tr.shape}')
print(f'Xp_tr: {Xp_tr.shape}')

## Datasets & DataLoaders

In [ ]:
class IMUDataset(Dataset):
    def __init__(self, X, M, y, reps, has_reps=None, augment=False):
        self.X        = torch.tensor(X.transpose(0, 2, 1), dtype=torch.float32)  # (N,6,MAX_LEN)
        self.M        = torch.tensor(M, dtype=torch.bool)                        # (N,MAX_LEN)
        self.y        = torch.tensor(y,    dtype=torch.long)
        self.reps     = torch.tensor(reps, dtype=torch.float32).unsqueeze(1)
        n = len(y)
        self.has_reps = torch.ones(n, dtype=torch.bool) if has_reps is None \
                        else torch.tensor(has_reps, dtype=torch.bool)
        self.augment  = augment

    def __len__(self):  return len(self.y)

    def __getitem__(self, i):
        x = self.X[i].clone()
        m = self.M[i]
        if self.augment:
            # Random amplitude scale ±20%
            x = x * (0.8 + 0.4 * torch.rand(1))
            # Small additive noise on real timesteps only
            noise = 0.04 * torch.randn_like(x)
            noise[:, ~m] = 0.0
            x = x + noise
            # No circular shift — raw signals preserve temporal structure
        return x, m, self.y[i], self.reps[i], self.has_reps[i]


BATCH = 64

pre_train_loader = None if len(yp_tr) == 0 else DataLoader(
    IMUDataset(Xp_tr,  Mp_tr,  yp_tr,  rp_tr,  hrp_tr,  augment=True),  BATCH, shuffle=True,  num_workers=0)
pre_val_loader   = None if len(yp_val) == 0 else DataLoader(
    IMUDataset(Xp_val, Mp_val, yp_val, rp_val, hrp_val, augment=False),  BATCH, shuffle=False, num_workers=0)

ft_train_loader  = DataLoader(
    IMUDataset(Xw_tr,  Mw_tr,  yw_tr,  rw_tr,  hrw_tr,  augment=True),  BATCH, shuffle=True,  num_workers=0)
ft_val_loader    = DataLoader(
    IMUDataset(Xw_val, Mw_val, yw_val, rw_val, hrw_val, augment=False),  BATCH, shuffle=False, num_workers=0)
ft_test_loader   = DataLoader(
    IMUDataset(Xw_te,  Mw_te,  yw_te,  rw_te,  hrw_te,  augment=False),  BATCH, shuffle=False, num_workers=0)

print('Loaders ready.')
print(f'Pretrain loaders active: {pre_train_loader is not None}')

## Model — SetWiseV4

Single-branch raw-signal architecture with padding mask support:
```
Input (B, 6, MAX_LEN)  +  mask (B, MAX_LEN)
  ConvStem      EEGNet-style depthwise separable conv
  MHABlock      Pre-norm multi-head self-attention (key_padding_mask for padded positions)
  DilatedResBlock  dilation=1
  DilatedResBlock  dilation=2
  MaskedAvgPool    average only over real (non-padded) timesteps
  rep_head (primary)  / class_head (auxiliary, weight=0.1)
```

In [ ]:
class LayerNorm1d(nn.Module):
    def __init__(self, C):
        super().__init__()
        self.norm = nn.LayerNorm(C)
    def forward(self, x):
        return self.norm(x.transpose(1, 2)).transpose(1, 2)


class ConvStem(nn.Module):
    def __init__(self, in_ch=6, F1=16, D=2, stem_kern=32, dropout=0.1):
        super().__init__()
        F2 = F1 * D
        self.temporal = nn.Sequential(
            nn.Conv1d(in_ch, F1, stem_kern, padding=stem_kern // 2, bias=False),
            LayerNorm1d(F1), nn.ELU(),
        )
        self.depthwise = nn.Sequential(
            nn.Conv1d(F1, F2, 1, groups=F1, bias=False),
            LayerNorm1d(F2), nn.ELU(), nn.Dropout(dropout),
        )
        self.pointwise = nn.Sequential(
            nn.Conv1d(F2, F2, 10, padding=5, bias=False),
            LayerNorm1d(F2), nn.ELU(), nn.Dropout(dropout),
        )
    def forward(self, x):
        return self.pointwise(self.depthwise(self.temporal(x)))


class MHABlock(nn.Module):
    """Pre-norm MHA with optional key_padding_mask for padded positions."""
    def __init__(self, d_model, num_heads=4, dropout=0.1):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, num_heads, dropout=dropout, batch_first=True)
        self.drop = nn.Dropout(dropout)
    def forward(self, x, key_padding_mask=None):
        # PyTorch key_padding_mask: True = position to ignore
        h, _ = self.attn(self.norm(x), self.norm(x), self.norm(x),
                         key_padding_mask=key_padding_mask)
        return x + self.drop(h)


class DilatedResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, k=4, dilation=1, dropout=0.3):
        super().__init__()
        p = (k - 1) * dilation
        self.pad1  = nn.ConstantPad1d((p, 0), 0)
        self.conv1 = nn.Conv1d(in_ch,  out_ch, k, dilation=dilation, bias=False)
        self.bn1   = nn.BatchNorm1d(out_ch)
        self.pad2  = nn.ConstantPad1d((p, 0), 0)
        self.conv2 = nn.Conv1d(out_ch, out_ch, k, dilation=dilation, bias=False)
        self.bn2   = nn.BatchNorm1d(out_ch)
        self.skip  = nn.Conv1d(in_ch, out_ch, 1, bias=False) if in_ch != out_ch else nn.Identity()
        self.drop  = nn.Dropout(dropout)
        self.act   = nn.ELU()
    def forward(self, x):
        h = self.drop(self.act(self.bn1(self.conv1(self.pad1(x)))))
        h = self.drop(self.act(self.bn2(self.conv2(self.pad2(h)))))
        return self.act(h + self.skip(x))


class SetWiseV4(nn.Module):
    """Single-branch raw-signal model with padding mask support."""
    def __init__(self, in_channels=N_CHANNELS, num_classes=10,
                 F1=16, D=2, stem_kern=32,
                 num_heads=4, attn_dropout=0.1,
                 tcn_ch=64, tcn_k=4, tcn_dropout=0.3,
                 head_dropout=0.4):
        super().__init__()
        F2 = F1 * D
        self.stem = ConvStem(in_channels, F1=F1, D=D,
                             stem_kern=stem_kern, dropout=attn_dropout)
        self.mha  = MHABlock(F2, num_heads=num_heads, dropout=attn_dropout)
        self.tcn1 = DilatedResBlock(F2,     tcn_ch, k=tcn_k, dilation=1, dropout=tcn_dropout)
        self.tcn2 = DilatedResBlock(tcn_ch, tcn_ch, k=tcn_k, dilation=2, dropout=tcn_dropout)

        self.rep_head = nn.Sequential(
            nn.Linear(tcn_ch, 32), nn.ReLU(),
            nn.Linear(32, 1)
        )
        self.class_head = nn.Sequential(
            nn.Linear(tcn_ch, 64), nn.ReLU(), nn.Dropout(head_dropout),
            nn.Linear(64, num_classes)
        )

    def _masked_pool(self, h, mask):
        """Average only over real (non-padded) timesteps. h: (B,C,T), mask: (B,T) bool."""
        m = mask.unsqueeze(1).float()   # (B,1,T)
        return (h * m).sum(-1) / m.sum(-1).clamp(min=1)  # (B,C)

    def _align_mask(self, mask, T_out):
        """Resize mask to T_out. ConvStem may expand T by 1-2 (symmetric even-kernel padding).
        Extra positions beyond the original signal are treated as padding (False)."""
        T_in = mask.shape[1]
        if T_out == T_in:
            return mask
        if T_out < T_in:
            return mask[:, :T_out]
        extra = torch.zeros(mask.shape[0], T_out - T_in, dtype=torch.bool, device=mask.device)
        return torch.cat([mask, extra], dim=1)

    def encode(self, x, mask=None):
        h   = self.stem(x)                                # (B, F2, T')
        kpm = None
        if mask is not None:
            mask = self._align_mask(mask, h.shape[-1])
            kpm  = ~mask                                  # PyTorch MHA: True = ignore
        h = self.mha(h.transpose(1, 2), key_padding_mask=kpm).transpose(1, 2)
        h = self.tcn1(h)                                  # causal padding preserves T'
        h = self.tcn2(h)
        if mask is not None:
            return self._masked_pool(h, mask)             # (B, tcn_ch)
        return h.mean(-1)

    def forward(self, x, mask=None):
        h = self.encode(x, mask)
        return self.class_head(h), self.rep_head(h)


model    = SetWiseV4(num_classes=len(label_names)).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f'Parameters: {n_params:,}')

## Training Utilities

In [ ]:
cls_loss_fn = nn.CrossEntropyLoss()
rep_loss_fn = nn.L1Loss()


def run_epoch(loader, optimizer=None):
    training = optimizer is not None
    model.train() if training else model.eval()
    total_loss = 0.0
    ys, yhats, rs, rhats, hrs = [], [], [], [], []

    for xb, mb, yb, rb, hrb in loader:
        xb, mb, yb, rb = xb.to(device), mb.to(device), yb.to(device), rb.to(device)
        hrb = hrb.to(device)
        with torch.set_grad_enabled(training):
            logits, rep_pred = model(xb, mb)
            cls_loss = cls_loss_fn(logits, yb)
            if hrb.any():
                rep_loss = rep_loss_fn(rep_pred[hrb], rb[hrb])
                loss = REP_WEIGHT * rep_loss + CLS_WEIGHT * cls_loss
            else:
                # No rep supervision in this batch — train classification at full weight
                loss = cls_loss
            if training:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
        total_loss += loss.item() * xb.size(0)
        ys.append(yb.cpu().numpy())
        yhats.append(logits.argmax(1).detach().cpu().numpy())
        rs.append(rb.cpu().numpy().ravel())
        rhats.append(rep_pred.detach().cpu().numpy().ravel())
        hrs.append(hrb.cpu().numpy())

    ys,   yhats = np.concatenate(ys),   np.concatenate(yhats)
    rs,   rhats = np.concatenate(rs),   np.concatenate(rhats)
    hrs         = np.concatenate(hrs)
    mae = mean_absolute_error(rs[hrs], rhats[hrs]) if hrs.any() else float('nan')

    return (total_loss / len(loader.dataset),
            (ys == yhats).mean(),
            mae,
            ys, yhats, rs, rhats)


def train_phase(train_loader, val_loader, epochs, lr, label, freeze_encoder=False):
    if freeze_encoder:
        for p in list(model.stem.parameters()) + \
                 list(model.mha.parameters())  + \
                 list(model.tcn1.parameters()) + \
                 list(model.tcn2.parameters()):
            p.requires_grad_(False)
        params = filter(lambda p: p.requires_grad, model.parameters())
        print(f'[{label}] Encoder frozen — training heads only')
    else:
        for p in model.parameters(): p.requires_grad_(True)
        params = model.parameters()
        print(f'[{label}] All parameters trainable')

    optimizer = torch.optim.AdamW(params, lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=lr/100)

    best_loss, best_state = float('inf'), None
    for epoch in range(epochs):
        tr = run_epoch(train_loader, optimizer)
        vl = run_epoch(val_loader)
        scheduler.step()
        if vl[0] < best_loss:
            best_loss  = vl[0]
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f'  [{label}] Ep {epoch+1:3d} | '
                  f'tr loss {tr[0]:.4f} rep-mae {tr[2]:.2f} acc {tr[1]:.3f} | '
                  f'vl loss {vl[0]:.4f} rep-mae {vl[2]:.2f} acc {vl[1]:.3f}')

    model.load_state_dict(best_state)
    print(f'  [{label}] Best val loss: {best_loss:.4f}\n')
    return best_state

## Phase 1 — Pretrain on recofit + MyoGym (forearm)

Rep head trains on recofit annotated sets (1 011 sets, range 1–61).
Cycle frequency in T_NORM input encodes rep count; transfers across sensor placements.

In [38]:
if pre_train_loader is None or pre_val_loader is None:
    print('=== Phase 1: Skipped (no recofit/MyoGym available) ===')
    pretrain_state = None
else:
    print('=== Phase 1: Pretrain on recofit + MyoGym ===')
    pretrain_state = train_phase(
        pre_train_loader, pre_val_loader,
        epochs=60, lr=1e-3,
        label='pretrain',
        freeze_encoder=False
    )

=== Phase 1: Pretrain on recofit + MyoGym ===
[pretrain] All parameters trainable
  [pretrain] Ep   1 | tr loss 19.3399 rep-mae 19.10 acc 0.130 | vl loss 16.7791 rep-mae 16.54 acc 0.224
  [pretrain] Ep  10 | tr loss 2.4589 rep-mae 2.23 acc 0.212 | vl loss 2.6400 rep-mae 2.42 acc 0.224
  [pretrain] Ep  20 | tr loss 2.1099 rep-mae 1.89 acc 0.224 | vl loss 3.3711 rep-mae 3.16 acc 0.224
  [pretrain] Ep  30 | tr loss 1.8762 rep-mae 1.66 acc 0.229 | vl loss 2.8393 rep-mae 2.63 acc 0.224
  [pretrain] Ep  40 | tr loss 1.6398 rep-mae 1.43 acc 0.235 | vl loss 2.7242 rep-mae 2.51 acc 0.217
  [pretrain] Ep  50 | tr loss 1.6058 rep-mae 1.39 acc 0.235 | vl loss 2.2617 rep-mae 2.04 acc 0.236
  [pretrain] Ep  60 | tr loss 1.5619 rep-mae 1.35 acc 0.241 | vl loss 2.8617 rep-mae 2.65 acc 0.236
  [pretrain] Best val loss: 2.0543



## Phase 2a — Fine-tune heads only (frozen encoder)

In [39]:
print('=== Phase 2a: Fine-tune heads (encoder frozen) ===')
train_phase(
    ft_train_loader, ft_val_loader,
    epochs=20, lr=1e-3,
    label='ft-heads',
    freeze_encoder=True
)

=== Phase 2a: Fine-tune heads (encoder frozen) ===
[ft-heads] Encoder frozen — training heads only
  [ft-heads] Ep   1 | tr loss 6.3878 rep-mae 6.07 acc 0.057 | vl loss 4.6175 rep-mae 4.32 acc 0.087
  [ft-heads] Ep  10 | tr loss 2.4866 rep-mae 2.25 acc 0.179 | vl loss 3.1407 rep-mae 2.90 acc 0.174
  [ft-heads] Ep  20 | tr loss 2.4737 rep-mae 2.24 acc 0.198 | vl loss 3.1331 rep-mae 2.90 acc 0.174
  [ft-heads] Best val loss: 3.1217



{'stem.temporal.0.weight': tensor([[[ 6.5298e-02,  5.5568e-02,  5.9302e-03,  ...,  3.1198e-02,
            4.8492e-02,  5.8408e-02],
          [-9.8763e-03, -3.2730e-02,  2.1038e-02,  ..., -7.0634e-02,
           -2.1046e-02, -8.8344e-02],
          [ 1.4128e-02, -4.4887e-02, -8.6812e-02,  ..., -1.5547e-02,
           -1.0261e-01, -7.5727e-02],
          [-8.2419e-03, -2.7094e-02, -2.5211e-02,  ..., -1.9822e-02,
            1.1070e-02, -4.3312e-02],
          [-6.2847e-02, -8.8500e-02, -3.6395e-02,  ...,  5.5495e-02,
           -5.0196e-02,  3.9743e-02],
          [-4.0537e-02,  5.1584e-02, -3.8987e-03,  ..., -2.6945e-03,
            9.5452e-02,  8.2694e-03]],
 
         [[-2.3018e-02, -8.2560e-02,  4.3085e-02,  ..., -6.9196e-02,
           -8.0331e-03, -3.3100e-02],
          [ 6.4449e-02,  3.3809e-02, -1.1323e-03,  ...,  9.0853e-02,
            3.4044e-02, -8.0429e-03],
          [-3.0450e-03, -6.6308e-02, -4.3142e-02,  ..., -7.8084e-03,
           -5.1643e-02, -3.1561e-02],
        

## Phase 2b — Fine-tune full model (lower LR)

In [40]:
print('=== Phase 2b: Full fine-tune on wrist data ===')
train_phase(
    ft_train_loader, ft_val_loader,
    epochs=50, lr=2e-4,
    label='ft-full',
    freeze_encoder=False
)

=== Phase 2b: Full fine-tune on wrist data ===
[ft-full] All parameters trainable
  [ft-full] Ep   1 | tr loss 2.4647 rep-mae 2.22 acc 0.208 | vl loss 2.9721 rep-mae 2.73 acc 0.174
  [ft-full] Ep  10 | tr loss 1.8652 rep-mae 1.63 acc 0.198 | vl loss 2.8585 rep-mae 2.62 acc 0.174
  [ft-full] Ep  20 | tr loss 1.6232 rep-mae 1.39 acc 0.217 | vl loss 3.0498 rep-mae 2.82 acc 0.174
  [ft-full] Ep  30 | tr loss 1.5257 rep-mae 1.30 acc 0.217 | vl loss 2.8333 rep-mae 2.60 acc 0.174
  [ft-full] Ep  40 | tr loss 1.5981 rep-mae 1.37 acc 0.208 | vl loss 2.8004 rep-mae 2.57 acc 0.174
  [ft-full] Ep  50 | tr loss 1.4780 rep-mae 1.25 acc 0.217 | vl loss 2.8731 rep-mae 2.64 acc 0.174
  [ft-full] Best val loss: 2.7893



{'stem.temporal.0.weight': tensor([[[ 6.6395e-02,  5.5822e-02,  6.6209e-03,  ...,  3.1523e-02,
            4.9066e-02,  5.7310e-02],
          [-1.3184e-02, -3.5144e-02,  1.8704e-02,  ..., -7.2259e-02,
           -2.0494e-02, -9.0409e-02],
          [ 1.5675e-02, -4.3995e-02, -8.8154e-02,  ..., -1.7829e-02,
           -1.0574e-01, -8.0869e-02],
          [-1.4509e-02, -2.8786e-02, -2.5661e-02,  ..., -2.0641e-02,
            7.6125e-03, -5.0595e-02],
          [-5.8323e-02, -8.2465e-02, -3.0063e-02,  ...,  5.9436e-02,
           -4.4956e-02,  4.6535e-02],
          [-3.5121e-02,  5.7092e-02,  7.6421e-04,  ..., -7.0519e-03,
            9.2297e-02,  4.0273e-03]],
 
         [[-1.8325e-02, -7.9286e-02,  4.6447e-02,  ..., -6.2624e-02,
           -1.6629e-03, -2.6864e-02],
          [ 6.4057e-02,  2.9541e-02, -6.3654e-03,  ...,  9.5158e-02,
            3.8499e-02, -1.3032e-02],
          [-6.5684e-04, -6.4868e-02, -4.0060e-02,  ..., -5.4918e-03,
           -4.8678e-02, -2.6073e-02],
        

## Evaluation on Wrist Test Set

In [41]:
te_loss, te_acc, te_mae, y_true, y_pred, r_true, r_pred = run_epoch(ft_test_loader)

r_true_hr = r_true[hrw_te]
r_pred_hr = r_pred[hrw_te]
r_pred_rounded = np.round(r_pred_hr)

mae       = mean_absolute_error(r_true_hr, r_pred_hr)
off_by_0  = np.mean(r_pred_rounded == r_true_hr)
off_by_1  = np.mean(np.abs(r_pred_rounded - r_true_hr) <= 1)
off_by_2  = np.mean(np.abs(r_pred_rounded - r_true_hr) <= 2)

print('=== Rep Counting — Test Set (Whales) ===')
print(f'  MAE          : {mae:.3f} reps')
print(f'  Exact match  : {off_by_0:.3f}  ({off_by_0*100:.1f}%)')
print(f'  Within ±1    : {off_by_1:.3f}  ({off_by_1*100:.1f}%)')
print(f'  Within ±2    : {off_by_2:.3f}  ({off_by_2*100:.1f}%)')
print()

# Per-exercise breakdown
print('Per-exercise MAE:')
for ex_idx, ex_name in enumerate(label_names):
    mask = hrw_te & (y_true == ex_idx)
    if mask.sum() == 0: continue
    ex_mae = mean_absolute_error(r_true[mask], r_pred[mask])
    print(f'  {ex_name:<30} n={mask.sum():>3}  MAE={ex_mae:.2f}')

=== Rep Counting — Test Set (Whales) ===
  MAE          : 3.732 reps
  Exact match  : 0.043  (4.3%)
  Within ±1    : 0.217  (21.7%)
  Within ±2    : 0.435  (43.5%)

Per-exercise MAE:
  bench_press                    n=  4  MAE=3.19
  bicep_curls                    n=  4  MAE=3.09
  lateral_raises                 n=  2  MAE=3.73
  pullups                        n=  4  MAE=3.82
  rows                           n=  3  MAE=1.89
  shoulder_press                 n=  1  MAE=0.68
  tricep_extensions              n=  5  MAE=6.32


## Class Distribution in Test Set

In [42]:
print('Test set distribution:')
for ex, cnt in sorted(Counter(label_names[i] for i in y_true).items(), key=lambda x: -x[1]):
    print(f'  {ex:<32} {cnt}')

Test set distribution:
  tricep_extensions                5
  bicep_curls                      4
  pullups                          4
  bench_press                      4
  rows                             3
  lateral_raises                   2
  shoulder_press                   1


## Example Inference

In [ ]:
model.eval()
print(f"{'#':>4}  {'true_exercise':<30} {'pred_exercise':<30}  true_reps  pred_reps")
print('-' * 90)
for i in range(min(15, len(yw_te))):
    xb = torch.tensor(Xw_te[i:i+1].transpose(0, 2, 1), dtype=torch.float32).to(device)
    mb = torch.tensor(Mw_te[i:i+1], dtype=torch.bool).to(device)
    with torch.no_grad():
        logits, rep_out = model(xb, mb)
    pred_cls = logits.argmax(1).item()
    true_reps = f'{rw_te[i]:.1f}' if hrw_te[i] else 'N/A'
    print(f"{i:>4}  {label_names[yw_te[i]]:<30} {label_names[pred_cls]:<30}  "
          f"{true_reps:>9}  {rep_out.item():>9.2f}")